# Session 5 — Project Tasks — P4A 2026
**Group 23**

This notebook implements the local LLM and data agent tasks for Session 5 of the group project.

## Setup
Initialize ChatOllama with the local `qwen3.5:9b` model and import required libraries.

In [ ]:
import os
import json
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, UserMessage, AIMessage
from langchain_core.tools import tool
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate

# Configure local LLM model
MODEL_NAME = "qwen3.5:9b"
print(f"Connecting to local Ollama with model: {MODEL_NAME}")

# Initialize ChatOllama
llm = ChatOllama(model=MODEL_NAME, temperature=0)


## Task 1 — Interpret findings with an LLM

Pass findings from Session 4 (Top Industries and Medical Insurance offerings) to the local LLM and interpret them.

In [ ]:
# Prepare findings from Session 4
findings = """
Finding 1: Top industries by full-time posting count:
1. Hospitals and Health Care: 4,557 postings
2. Financial Services: 2,394 postings
3. Retail: 2,383 postings
4. Staffing and Recruiting: 2,334 postings

Finding 2: Medical Insurance benefits by industry (Top industries offering Medical Insurance):
1. Hospitals and Health Care: 235 postings
2. Staffing and Recruiting: 130 postings
3. Financial Services: 121 postings
4. Construction: 116 postings

Note: Retail is completely absent from the top 10 list for medical insurance offerings, despite being the 3rd largest industry by overall postings.
"""

system_prompt = SystemMessage(
    content="You are a labor economist and business strategist. Your goal is to analyze job posting data and extract meaningful, non-trivial insights."
)

user_prompt = UserMessage(
    content=f"Analyze the following findings from our LinkedIn Job Postings dataset. Specifically, interpret why the Retail industry has such a high listing volume but is completely absent from the top 10 list of industries offering Medical Insurance benefits:\n\n{findings}"
)

messages = [system_prompt, user_prompt]
response = llm.invoke(messages)

print("--- Data Passed to Model ---")
print(findings)
print("\n--- Prompts Used ---")
print(f"System Prompt: {system_prompt.content}")
print(f"User Prompt: {user_prompt.content}")
print("\n--- Model's Response ---")
print(response.content)


**Value Assessment:**
Does the response add value beyond what the data already shows, or is it just restating the numbers?

*Replace this text with your assessment.*

## Task 2 — Structured output

Ask the LLM to summarize the retail vs. healthcare benefits discrepancy into a structured JSON object.

In [ ]:
structured_prompt = f"""
Summarize the discrepancy between high posting counts and low medical insurance offerings in the Retail industry from our dataset.
You must return the output as a valid JSON object matching the following schema. Do not return any other text or markdown block, only the JSON itself.

{{
  "highest_posting_industry": "string",
  "discrepancy_industry": "string",
  "observed_gap_description": "string",
  "proposed_reason": "string",
  "actionability_level": "string (High/Medium/Low)"
}}

Findings:
{findings}
"""

response_structured = llm.invoke([UserMessage(content=structured_prompt)])
raw_response = response_structured.content.strip()
print("--- Raw Model Response ---")
print(raw_response)

# Strip markdown backticks if present
if raw_response.startswith("```json"):
    raw_response = raw_response[7:]
elif raw_response.startswith("```"):
    raw_response = raw_response[3:]
if raw_response.endswith("```"):
    raw_response = raw_response[:-3]
raw_response = raw_response.strip()

print("\n--- Parsed Output ---")
try:
    parsed_json = json.loads(raw_response)
    print(json.dumps(parsed_json, indent=2))
except Exception as e:
    print(f"Failed to parse JSON: {e}")


**JSON Quality Assessment:**
Did the model return valid JSON on the first try? If not, what did you do?

*Replace this text with your assessment.*

## Task 3 — Multi-turn conversation

Hold a 3-turn conversation about findings, building context as the turns progress.

In [ ]:
conversation_history = [
    SystemMessage(content="You are a helpful data science assistant guiding a business analyst through LinkedIn job postings research.")
]

turns = [
    "Based on our job postings dataset, what are the top industries by volume of listings, and which of these are the most generous in providing medical insurance?",
    "Interesting. Why do you think Retail has so many job postings but almost no postings listing medical insurance compared to Hospitals and Health Care or Financial Services?",
    "Given this benefit gap in retail, how can we use this data to advise a job board or recruitment agency on how to target employer outreach or improve listing conversions?"
]

for i, turn in enumerate(turns, 1):
    print(f"\n=== Turn {i} ===")
    print(f"User: {turn}")
    conversation_history.append(UserMessage(content=turn))
    
    response = llm.invoke(conversation_history)
    print(f"AI: {response.content}")
    conversation_history.append(AIMessage(content=response.content))

# Print message history summary
print("\n=== Message History Summary ===")
for msg in conversation_history:
    role = "System" if isinstance(msg, SystemMessage) else ("User" if isinstance(msg, UserMessage) else "AI")
    preview = msg.content[:80].replace("\n", " ") + "..."
    print(f"- {role}: {preview}")


**Context Assessment:**
Did the model maintain context correctly across the three turns?

*Replace this text with your assessment.*

## Task 4 — Build a data agent

Define custom tools that query the output files directly and run a ReAct agent to demonstrate reasoning.

In [ ]:
@tool
def get_industry_posting_count(industry_name: str) -> str:
    """Queries the dataset and returns the number of full-time postings for the given industry name.
    Useful for answering questions about industry posting volume.
    """
    try:
        df = pd.read_csv("output/industry_posting_counts.csv")
        match = df[df['industry_name'].str.lower() == industry_name.lower()]
        if not match.empty:
            count = match.iloc[0]['posting_count']
            return f"Industry '{industry_name}' has {count} full-time job postings."
        else:
            return f"Industry '{industry_name}' was not found in the dataset."
    except Exception as e:
        return f"Error querying dataset: {str(e)}"

@tool
def get_top_industries(n: int = 5) -> str:
    """Returns the top N industries by number of full-time postings.
    Useful for answering questions about which industries are the largest by volume.
    """
    try:
        df = pd.read_csv("output/industry_posting_counts.csv")
        top_n = df.head(n)
        result = "Top industries by posting count:\n"
        for i, row in top_n.iterrows():
            result += f"{i+1}. {row['industry_name']}: {row['posting_count']} postings\n"
        return result
    except Exception as e:
        return f"Error loading top industries: {str(e)}"

@tool
def get_top_countries(n: int = 5) -> str:
    """Returns the top N countries by number of companies in the dataset.
    Useful for answering questions about the geographic distribution of companies.
    """
    try:
        with open("output/top_countries.json", "r") as f:
            data = json.load(f)
        top_countries = data.get("top_countries", [])[:n]
        result = "Top countries by number of companies:\n"
        for i, country in enumerate(top_countries):
            result += f"{i+1}. {country['country']}: {country['count']} companies\n"
        return result
    except Exception as e:
        return f"Error loading top countries: {str(e)}"

# List of tools
tools = [get_industry_posting_count, get_top_industries, get_top_countries]

# Build the ReAct Agent Prompt
template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

# Initialize ReAct Agent
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

# Test agent trace
question = "How many full-time postings does the Financial Services industry have, and which are the top 3 countries by company count in our dataset?"
print(f"Question: {question}\n")
response = agent_executor.invoke({"input": question})
print("\nFinal Answer:\n", response["output"])
